In [5]:
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — Runtime → T4")
assert torch.cuda.is_available(), "Set runtime to T4 GPU"

%pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pillow
%pip -q install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
%pip -q install --no-deps --upgrade "torchao>=0.16.0"
%pip -q install transformers==4.57.6
%pip -q install --no-deps trl==0.22.2

from unsloth import FastVisionModel
print("unsloth ok")

cuda True Tesla T4
Unsloth: xformers already installed and loads.
Unsloth: skipping flash_attn, which needs sm80 or newer (the best GPU is sm75).
Unsloth: causal_conv1d already installed and loads.
Unsloth: mamba_ssm already installed and loads.
unsloth ok


data fetching

In [6]:
from pathlib import Path

DATA = Path("/content/dataset")
DATA.mkdir(exist_ok=True)
cands = list(DATA.rglob("sequences.json"))
ZIP = None

if not cands:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("drive skip", type(e).__name__, e)

    hits = []
    for root in (Path("/content/drive/MyDrive") if Path("/content/drive/MyDrive").exists() else Path("/content"), Path("/content")):
        if root.exists():
            hits.extend(root.glob("**/*manga*.zip"))
            hits.extend(root.glob("**/byte_manga.zip"))
    hits = [p for p in hits if p.is_file()]
    print("zip hits", hits[:8])
    ZIP = hits[0] if hits else Path("/content/byte_manga.zip")
    assert ZIP.exists(), "Upload byte_manga.zip to Drive as MyDrive/byte_manga.zip OR Files pane → /content/byte_manga.zip"
    !unzip -q "{ZIP}" -d /content/dataset
    cands = list(DATA.rglob("sequences.json"))

assert cands, "unzip did not produce sequences.json"
DATA = cands[0].parent
print("DATA =", DATA)
print("labels", (DATA / "development" / "labels.jsonl").exists(), "score", (DATA / "score.py").exists(), "n png", len(list(DATA.rglob("*.png"))))
assert (DATA / "development" / "labels.jsonl").is_file()
assert (DATA / "score.py").is_file()

DATA = /content/dataset/BYTE ML Task Data
labels True score True n png 285


##  train

In [7]:
import json, random
from pathlib import Path

PROMPT = (
    "You see 3 consecutive English manga pages (page 1, then 2, then 3).\n"
    "Return ONLY JSON with this shape:\n"
    '{"pages":[[{"speaker":"character1","text":"..."}],[],[]]}\n\n'
    "Include: dialogue, thoughts, narration, clear vocalisations, punctuation balloons (... ?!).\n"
    "Drop: SFX and translated SFX (SNAP, BAM, SFX: HOOFBEATS), titles/logos/credits/page numbers/ads, "
    "LAST TIME / recap boxes, editor notes, signs/clothes/objects, text inside letters/phones/news, tiny breaths.\n"
    "Speakers: character1, character2, ... Same person keeps the same id across all 3 pages. "
    "Narrator boxes = NARRATION. If you cannot tell = UNKNOWN. Thoughts belong to that character, not NARRATION.\n"
    "Reading order is usually right-to-left, top-to-bottom; follow panel/balloon flow. "
    "Keep stutters. Join wrap-only newlines. Empty page = []. Do not invent text."
)

def load_jsonl(path):
    rows = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            o = json.loads(line)
            rows[str(o["sequence_id"])] = o
    return rows

def load_sequences(root):
    raw = json.loads((root / "sequences.json").read_text(encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [{"sequence_id": k, **(v if isinstance(v, dict) else {"images": v})} for k, v in raw.items()]
    items = []
    for v in raw:
        items.append({
            "sequence_id": str(v.get("sequence_id") or v.get("id")),
            "split": v.get("split"),
            "images": v.get("images") or v.get("pages") or v.get("files") or [],
        })
    return items

def dump_pages(pages):
    return json.dumps({"pages": pages}, ensure_ascii=False, separators=(",", ":"))

def split_dev_ids(dev_ids, val_frac=0.2, seed=13):
    ids = list(dev_ids)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n_val = max(1, int(round(len(ids) * val_frac)))
    return sorted(ids[n_val:]), sorted(ids[:n_val])

def write_split(root, out_dir, seed=13):
    root = Path(root)
    seqs = load_sequences(root)
    labels = load_jsonl(root / "development" / "labels.jsonl")
    sample = load_jsonl(root / "sample_submission.jsonl")
    by_id = {s["sequence_id"]: s for s in seqs}
    train_ids, val_ids = split_dev_ids(list(labels), seed=seed)
    test = [by_id[i] for i in sample if i in by_id] or [s for s in seqs if s["sequence_id"] not in labels]

    def pack(seq, part):
        sid = seq["sequence_id"]
        rel = [str(p) for p in (seq.get("images") or [])[:3]]
        while len(rel) < 3:
            rel.append("")
        gold = (labels.get(sid) or {}).get("pages") or [[], [], []]
        while len(gold) < 3:
            gold.append([])
        return {"sequence_id": sid, "part": part, "images": rel[:3], "pages": gold[:3]}

    recs = {
        "train": [pack(by_id[i], "train") for i in train_ids if i in by_id],
        "val": [pack(by_id[i], "val") for i in val_ids if i in by_id],
        "test": [{"sequence_id": s["sequence_id"], "part": "test", "images": (s.get("images") or [])[:3], "pages": [[], [], []]} for s in test],
    }
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for part, rows in recs.items():
        with (out_dir / f"{part}.jsonl").open("w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    (out_dir / "split.json").write_text(json.dumps({k: [r["sequence_id"] for r in v] for k, v in recs.items()}, indent=2), encoding="utf-8")
    return {k: len(v) for k, v in recs.items()}

if Path("/content/drive/MyDrive").exists():
    OUT = Path("/content/drive/MyDrive/byte_manga_work_unsloth")
else:
    OUT = Path("/content/byte_manga_work_unsloth")
OUT.mkdir(parents=True, exist_ok=True)
counts = write_split(DATA, OUT / "sft", seed=13)
print("OUT", OUT)
print("split", counts)
assert counts.get("train") == 64 and counts.get("val") == 16, counts

train = [json.loads(l) for l in (OUT / "sft" / "train.jsonl").read_text().splitlines() if l.strip()]
val = [json.loads(l) for l in (OUT / "sft" / "val.jsonl").read_text().splitlines() if l.strip()]
test = [json.loads(l) for l in (OUT / "sft" / "test.jsonl").read_text().splitlines() if l.strip()]
print("n train/val/test", len(train), len(val), len(test))

OUT /content/drive/MyDrive/byte_manga_work_unsloth
split {'train': 64, 'val': 16, 'test': 15}
n train/val/test 64 16 15


In [8]:
from unsloth import FastVisionModel
from PIL import Image
import torch

SKIP_TRAIN = False
MAX_PIXELS = 384 * 28 * 28
MODEL_ID = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
ADAPTER = OUT / "adapter"

def load_image(path, max_pixels=MAX_PIXELS):
    im = Image.open(path).convert("RGB")
    w, h = im.size
    pix = w * h
    if pix > max_pixels:
        scale = (max_pixels / pix) ** 0.5
        im = im.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.Resampling.BICUBIC)
    return im

if SKIP_TRAIN:
    assert ADAPTER.exists(), ADAPTER
    model, tokenizer = FastVisionModel.from_pretrained(str(ADAPTER), load_in_4bit=True, dtype=torch.float16)
    FastVisionModel.for_inference(model)
    print("loaded adapter", ADAPTER)
else:
    model, tokenizer = FastVisionModel.from_pretrained(
        MODEL_ID,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
        dtype=torch.float16,
    )
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=8,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=13,
        use_rslora=False,
        loftq_config=None,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    try:
        model.print_trainable_parameters()
    except Exception:
        n = sum(p.numel() for p in model.parameters() if p.requires_grad)
        t = sum(p.numel() for p in model.parameters())
        print(f"trainable {n}/{t} ({100 * n / max(t, 1):.4f}%)")

==((====))==  Unsloth 2026.10.3: Fast Qwen2_Vl patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
trainable params: 9,232,384 || all params: 2,218,217,984 || trainable%: 0.4162


In [9]:
def to_conversations(records):
    out = []
    for rec in records:
        paths = [p for p in (DATA / rel for rel in rec["images"] if rel) if p.is_file()]
        if len(paths) < 3:
            print("skip", rec.get("sequence_id"))
            continue
        images = [load_image(p) for p in paths[:3]]
        content = [{"type": "image", "image": im} for im in images]
        content.append({"type": "text", "text": PROMPT})
        out.append({
            "messages": [
                {"role": "user", "content": content},
                {"role": "assistant", "content": [{"type": "text", "text": dump_pages(rec["pages"])}]},
            ]
        })
    return out

converted = to_conversations(train)
print("conversations", len(converted))
assert converted, "no train images — DATA path wrong"

conversations 64


In [10]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from unsloth import FastVisionModel

if SKIP_TRAIN:
    print("skip train")
else:
    FastVisionModel.for_training(model)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer, resize="max"),
        train_dataset=converted,
        args=SFTConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            num_train_epochs=2,
            learning_rate=1e-4,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=13,
            output_dir=str(OUT / "hf_trainer"),
            report_to="none",
            fp16=True,
            bf16=False,
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=4096,
        ),
    )
    trainer.train()
    ADAPTER.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(ADAPTER))
    tokenizer.save_pretrained(str(ADAPTER))
    print("saved", ADAPTER)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 64 | Num Epochs = 2 | Total steps = 32
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 9,232,384 of 2,218,217,984 (0.42% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.013100
2,1.911600
3,2.128600
4,1.973700
5,1.625000
6,1.957700
7,1.508100
8,1.933100
9,1.485300
10,1.795100


saved /content/drive/MyDrive/byte_manga_work_unsloth/adapter


In [11]:
import re, json, subprocess, sys
from unsloth import FastVisionModel

FastVisionModel.for_inference(model)

def parse_pages(text):
    text = text.strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return [[], [], []]
    try:
        o = json.loads(m.group(0))
    except json.JSONDecodeError:
        return [[], [], []]
    pages = o.get("pages") or o.get("page")
    if isinstance(pages, list) and pages and isinstance(pages[0], dict):
        pages = [pages, [], []]
    if not isinstance(pages, list):
        pages = [[], [], []]
    out = []
    for page in pages[:3]:
        cleaned = []
        if not isinstance(page, list):
            out.append([])
            continue
        for row in page:
            if not isinstance(row, dict):
                continue
            t = str(row.get("text") or "").replace("\n", " ").strip()
            sp = str(row.get("speaker") or "UNKNOWN").strip() or "UNKNOWN"
            if t:
                cleaned.append({"speaker": sp, "text": t})
        out.append(cleaned)
    while len(out) < 3:
        out.append([])
    return out[:3]

def generate_one(img_paths, max_new=1400):
    images = [load_image(p) for p in img_paths]
    messages = [{"role": "user", "content": [{"type": "image"} for _ in images] + [{"type": "text", "text": PROMPT}]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(images, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new, do_sample=False, use_cache=True)
    prompt_len = inputs["input_ids"].shape[1]
    decode = getattr(tokenizer, "decode", None) or tokenizer.tokenizer.decode
    text = decode(out_ids[0, prompt_len:], skip_special_tokens=True)
    return parse_pages(text), text

def infer_split(records):
    rows = []
    for rec in records:
        paths = [p for p in (DATA / rel for rel in rec["images"] if rel) if p.is_file()]
        if len(paths) < 3:
            pages, raw = [[], [], []], "missing-images"
        else:
            try:
                pages, raw = generate_one(paths)
            except Exception as e:
                print("gen fail", rec["sequence_id"], type(e).__name__, e)
                pages, raw = [[], [], []], "error"
        rows.append({"sequence_id": rec["sequence_id"], "pages": pages})
        print("pred", rec["sequence_id"], [len(p) for p in pages], str(raw)[:120].replace("\n", " "))
    return rows

def write_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("wrote", path, "n=", len(rows))

pred_dir = OUT / "predictions"
log_dir = OUT / "logs"
pred_dir.mkdir(exist_ok=True)
log_dir.mkdir(exist_ok=True)

val_rows = infer_split(val)
write_jsonl(val_rows, pred_dir / "val.jsonl")
labels = load_jsonl(DATA / "development" / "labels.jsonl")
val_gold = OUT / "sft" / "val_gold.jsonl"
write_jsonl([labels[r["sequence_id"]] for r in val_rows if r["sequence_id"] in labels], val_gold)
score_out = log_dir / "scores_val_unsloth.json"
cmd = [sys.executable, str(DATA / "score.py"), "--references", str(val_gold), "--predictions", str(pred_dir / "val.jsonl"), "--output", str(score_out)]
print("+", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:] if r.stdout else "")
if r.returncode:
    print(r.stderr[-2000:])
sc = json.loads(score_out.read_text()) if score_out.exists() else {}
macro = sc.get("macro") if isinstance(sc.get("macro"), dict) else sc
print("VAL", {k: macro.get(k) for k in ("text_order_score", "balanced_joint_f1")})

test_rows = infer_split(test)
sample = load_jsonl(DATA / "sample_submission.jsonl")
if sample:
    by = {row["sequence_id"]: row for row in test_rows}
    test_rows = [by[i] for i in sample if i in by] + [row for row in test_rows if row["sequence_id"] not in sample]
write_jsonl(test_rows, pred_dir / "test.jsonl")
print("DONE", ADAPTER)

pred seq_1c1d4f5e71a2f554 [1, 0, 0] {"pages":[[{"speaker":"character1","text":"IT FEELS LIKE IT'S ALIVE!! IT'S RAW FISH! ARE YOU KIDDING ME? IT'S ONLY FISH,
pred seq_2032620aa4e4ac7f [0, 0, 0] {"pages":[[{"speaker":"character1","text":"A STRONG AREA AND A MEAN FASTBALL! THIS IS DAIRY YOUTH GRAFFITI!"},{"speaker"
pred seq_27e88136c1812f85 [3, 0, 0] {"pages":[[{"speaker":"character1","text":"..."},{"speaker":"character1","text":"..."},{"speaker":"character1","text":".
pred seq_5c5304b3f7796f8c [3, 2, 2] {"pages":[[{"speaker":"character1","text":"AND HE CAN'T FIGHT NOW."},{"speaker":"character1","text":"HOW'S THAT POSSIBLE
pred seq_720a7023d4f2ffcb [1, 0, 0] {"pages":[[{"speaker":"character1","text":"..."}],[],[]]}
pred seq_7db7a000adeceacf [0, 0, 0] {"pages":[[{"speaker":"character1","text":"IF YOU WANT TO PUT IT ANOTHER WAY, YOU'RE EASY TO READ."},{"speaker":"charact
pred seq_83ff2f28f5c13c40 [0, 0, 0] {"pages":[[{"speaker":"character1","text":"IT'S ALL I'VE BEEN LISTENING TO YOU FOR YE